# GUNDAM interface demo

This notebook configures GUNDAM through `gundam-interface`, initializes the fitter engine, evaluates the nominal likelihood, runs a minimization, and inspects the best-fit parameters.


In [1]:
nCpuThreads = 3
gundamLibPath = "/Users/nadrino/Documents/Work/Install/gundam/lib"
workDir = "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024"
configPath = "configOA2024.yaml"
overrideList = [
    "override/v12ProdRun45.yaml",
    "override/onlyFlux5.yaml",
    "override/noEigen.yaml",
]
dataType = "Asimov"  # "Asimov", "Toy", or "RealData"
seed = 12345

In [2]:
import sys
import numpy as np
from pathlib import Path

# Prefer the local checkout when running this notebook before pip installation.
# If you install this package with pip, you can remove this block.
# User configuration
repoRoot = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
srcPath = repoRoot / "src"
if srcPath.exists() and str(srcPath) not in sys.path:
    sys.path.insert(0, str(srcPath))
# ~ end of this block

from gundam_interface import GundamLoader, GundamRuntime, GundamInterface


In [3]:
np.random.seed(seed)

runtime = GundamRuntime(
    loader=GundamLoader(gundamLibPath=gundamLibPath),
    workDir=workDir,
    nCpuThreads=nCpuThreads,
    configPath=configPath,
    overrideList=overrideList,
    dataType=dataType,
    randomSeed=seed,
)

runtime.toDict(includeConfigJsonString=False)


{'nCpuThreads': 3,
 'workDir': '/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024',
 'dataType': 'Asimov',
 'loader': {'moduleName': 'GUNDAM',
  'gundamLibPath': '/Users/nadrino/Documents/Work/Install/gundam/lib'},
 'randomSeed': 12345,
 'configPath': 'configOA2024.yaml',
 'overrideList': ['override/v12ProdRun45.yaml',
  'override/onlyFlux5.yaml',
  'override/noEigen.yaml']}

In [4]:
gundam = GundamInterface(runtime)
gundam.configure()

2026.07.28 14:19:35  INFO ConfigUtils: Reading config file: /Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/configOA2024.yaml
2026.07.28 14:19:35  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/v12ProdRun45.yaml"
2026.07.28 14:19:35  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/mc/filePathList
2026.07.28 14:19:35  WARN ConfigUtils:   Added: fitterEngineConfig/propagatorConfig/dataSetList/0(name:"ND280")/data/0(name:"data")/filePathList
2026.07.28 14:19:35  INFO ConfigUtils: Overriding config with "/Users/nadrino/Documents/Work/Output/results/gundam/GundamInputOA2024/override/onlyFlux5.yaml"
2026.07.28 14:19:35  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSetListConfig/0(name:"Flux Systematics")/isEnabled: true -> false
2026.07.28 14:19:35  WARN ConfigUtils:   Override: fitterEngineConfig/propagatorConfig/parameterSe

In [5]:
# runtime.logRedirector.debug = True
gundam.initialize()

2026.07.28 14:19:35 ALERT ConfigUtils: /fitterEngineConfig/propagatorConfig: key "showEventBreakdownBeforePrior" was ignored while parsing config. Is is context dependent option?
2026.07.28 14:19:35  INFO ParameterSet: Stripping the matrix from fixed/disabled parameters...
2026.07.28 14:19:35  INFO ParameterSet: 5 effective parameters were defined in set: Flux Systematics
2026.07.28 14:19:35  INFO ParameterSet: Computing inverse of the stripped covariance matrix: 5x5
2026.07.28 14:19:35  INFO ParametersManager: 5 enabled parameters in Flux Systematics
2026.07.28 14:19:35  INFO ParametersManager: Total number of parameters: 5
2026.07.28 14:19:35  INFO ParametersManager: Building global covariance matrix (5x5)
2026.07.28 14:19:35  INFO ParameterSet: Stripping the matrix from fixed/disabled parameters...
2026.07.28 14:19:35  INFO ParameterSet: 5 effective parameters were defined in set: Flux Systematics
2026.07.28 14:19:35  INFO ParameterSet: Computing inverse of the stripped covariance m

In [6]:
print(f"Initialized GUNDAM with {len(gundam.parameters)} active parameters")
for index, parameter in enumerate(gundam.parameters):
    print(
        f"{index:3d}: {parameter.name} "
        f"prior={parameter.prior:.8g} "
        f"step={parameter.stepSize:.8g} "
        f"value={parameter.value:.8g}"
    )

AttributeError: 'GundamInterface' object has no attribute 'parameters'

In [ ]:
nominalPhysicalValues = gundam.getParametersManager().getParameterValues()
nominalLlh = gundam.evaluateLlh()

print(f"Nominal LLH: {nominalLlh:.8g}")
print("Nominal parameters:")
for name, physicalValue in zip(gundam.parameterNames, nominalPhysicalValues):
    print(f"  - {name}: physical={physicalValue:.8g}")


In [ ]:
bestFitLlh = gundam.minimize()
bestFitPhysicalValues = gundam.getParametersManager().getParameterValues()
deltaLlh = bestFitLlh - nominalLlh

print(f"Best-fit LLH: {bestFitLlh:.8g}")
print(f"Delta LLH relative to nominal: {deltaLlh:.8g}")


In [ ]:
print("Best-fit parameters:")
for name, physicalValue in zip(gundam.parameterNames, bestFitPhysicalValues):
    print(f"  - {name}: physical={physicalValue:.8g}")


In [ ]:
# Re-evaluate the likelihood at the best-fit point as a consistency check.
reevaluatedBestFitLlh = gundam.evaluateLlh(physicalValues=bestFitPhysicalValues)

print(f"Minimizer best-fit LLH: {bestFitLlh:.8g}")
print(f"Re-evaluated best-fit LLH: {reevaluatedBestFitLlh:.8g}")
print(f"Absolute difference: {abs(reevaluatedBestFitLlh - bestFitLlh):.8g}")
